In [ ]:
import json
from typing import List
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from langchain_core.documents import Document
from langchain_chroma import Chroma
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()

True

In [20]:
MODEL = "moonshotai/Kimi-K2.6"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-8B"
INFERENCE_PROVIDER = "scaleway"


In [21]:
import os
os.environ["PATH"] += r";C:\Program Files\Tesseract-OCR"

In [22]:
def partition_document(file_path: str):
    print(f"Partitioning document: {file_path}")

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )

    print(f"Extracted {len(elements)} elements")
    return elements

In [23]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("Creating smart chunks...")

    chunks = chunk_by_title(
        elements,
        max_characters = 3000,
        new_after_n_chars = 2400,
        combine_text_under_n_chars=500
    )

    print(f"Created {len(chunks)} chunks")
    return chunks

In [24]:
def seperate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        "text": chunk.text,
        "tables": [],
        "images": [],
        "types": ["text"]
    }

    if hasattr(chunk, "metadata") and hasattr(chunk.metadata, "orig_elements"):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            if element_type == "Table":
                content_data['types'].append("table")
                table_html = getattr(element.metadata, "text_as_html", element.text)
                content_data['tables'].append(table_html)

            elif element_type == "Image":
                content_data['types'].append('image')
                if hasattr(element, 'metadata') and hasattr(element.metadata, "image_base64"):
                    content_data['images'].append(element.metadata.image_base64)
        
        content_data['types'] = list(set(content_data['types']))
        return content_data

In [ ]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]):
    """Create AI-enhanced summary for mixed content"""

    try:
        client = InferenceClient(
            provider = "fireworks-ai",
            api_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")
        )

        prompt_text = f"""You are creating a searchable description for document content retrieval.

                CONTENT TO ANALYZE:
                TEXT CONTENT:
                {text}

                """
        if tables:
            prompt_text += "Tables:\n"

            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        message_content = [{"type": "text", "text": prompt_text}]

        for image_base64 in images:
            message_content.append({"type": "image_url", "url": f"data:image/jpeg;base64,{image_base64}"})

        completion = client.chat.completions.create(
            model=MODEL,
            messages=[
            {
                "role": "user",
                "content": message_content
            }
            ],
            max_tokens=800,
            temperature=0.2,
            extra_body={"thinking": {"type": "disabled"}}
        )

        return completion.choices[0].message.content

    except Exception as e:
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [26]:
def summarize_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("Processing chunks with AI Summaries...")

    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")

        content_data = seperate_content_types(chunk)

        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        if content_data['tables'] or content_data['images']:
            print(f"     -> Creating AI summary for mixed content...")

            try:
                enhanced_content = create_ai_enhanced_summary(
                    text=content_data['text'],
                    tables=content_data['tables'],
                    images=content_data['images']
                )

                print(f"     -> AI summary created successfully")
                print(f"     -> Enhanced content preview: {enhanced_content[:200]}...")

            except Exception as e:
                print(f"AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     -> Using raw text (no tables/images)")
            enhanced_content = content_data['text']

        doc = Document(
            page_content=enhanced_content,
            metadata={
                'original_content': json.dumps({
                    'raw_text': content_data['text'],
                    'tables_html': content_data['tables'],
                    'images_base64': content_data['images']
                })
            }
        )

        langchain_documents.append(doc)

    print(f"Processed {len(langchain_documents)} chunks")
    return langchain_documents

In [27]:
import numpy as np
from langchain_core.embeddings import Embeddings

class HuggingFaceInferenceEmbeddings(Embeddings):
    def __init__(self, client, model):
        self.client = client
        self.model = model

    def _process(self, raw):
        arr = np.array(raw)
        if arr.ndim == 3:
            arr = arr[0]        # (1, seq, dim) → (seq, dim)
        if arr.ndim == 2:
            arr = arr[-1]       # last-token (EOS) pooling: (seq, dim) → (dim,)
        return arr.tolist()

    def embed_documents(self, texts):
        return [self._process(self.client.feature_extraction(text, model=self.model)) for text in texts]

    def embed_query(self, text):
        return self._process(self.client.feature_extraction(text, model=self.model))

def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("Creating embeddings and storing in ChromaDB...")

    client = InferenceClient(
        provider=INFERENCE_PROVIDER,
        api_key=os.getenv("HUGGINGFACEHUB_API_TOKEN")
    )

    embedding_model = HuggingFaceInferenceEmbeddings(client=client, model=EMBEDDING_MODEL)

    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")

    print(f"Vector store created and saved to {persist_directory}")
    return vectorstore


In [ ]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("Starting RAG Ingestion Pipeline")
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation (if text + tables/images, create enhanced summary else use raw text)
    summarised_chunks = summarize_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="db/chroma_db")
    
    print("Pipeline completed successfully!")
    return db

In [ ]:
run_complete_ingestion_pipeline("./docs/diabetes_6.pdf")

Uncomment and run below cell if you want to run complete ingestion pipeline for all pdfs in /docs folder in parallel

In [ ]:
# import glob
# from concurrent.futures import ThreadPoolExecutor, as_completed

# def process_single_pdf(pdf_path: str):
#     print(f"Processing: {pdf_path}")
#     elements = partition_document(pdf_path)
#     chunks = create_chunks_by_title(elements)
#     summarised_chunks = summarize_chunks(chunks)
#     print(f"Done: {pdf_path}")
#     return summarised_chunks

# pdf_files = glob.glob("./docs/*.pdf")
# print(f"Found {len(pdf_files)} PDF(s): {pdf_files}")

# all_documents = []

# with ThreadPoolExecutor(max_workers=len(pdf_files)) as executor:
#     futures = {executor.submit(process_single_pdf, pdf): pdf for pdf in pdf_files}
#     for future in as_completed(futures):
#         pdf = futures[future]
#         try:
#             docs = future.result()
#             all_documents.extend(docs)
#         except Exception as e:
#             print(f"Failed to process {pdf}: {e}")

# print(f"\nTotal documents to store: {len(all_documents)}")
# db = create_vector_store(all_documents, persist_directory="db/chroma_db")
# print("\nAll PDFs processed and stored in a single DB!")
